# Lab 03: Forward Dynamics — Build the DynamicsEngine

**Computational Sensorimotor Control** | Week 3 | Module 1: The Biological Plant

---

## Overview

In this lab you build the **DynamicsEngine** — the component that connects muscles to movement. Given muscle activations, it computes joint torques, solves the equations of motion M(q)d²q/dt² = τ − C(q,dq/dt)dq/dt, and integrates forward in time to produce joint angle trajectories.

By the end, you will simulate the arm responding to muscle activations and observe **interaction torques** — forces at one joint caused by motion at another.

**Structure:** Build standalone functions first (Parts 1–4), assemble into a class (Part 5), then run simulations (Parts 6–8).

---
## Part 0: Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Inertial parameters (Gribble et al. 1998)
l1, l2 = 0.34, 0.46       # segment lengths (m)
m1, m2 = 2.10, 1.65       # segment masses (kg)
s1, s2 = 0.11, 0.16       # COM distances from proximal joint (m)
I1, I2 = 0.015, 0.022     # moments of inertia about COM (kg·m²)

# Inertial constants (derived in lecture notes, Eq. 3.5)
a1 = I1 + I2 + m1*s1**2 + m2*(l1**2 + s2**2)
a2 = I2 + m2*s2**2
a3 = m2*l1*s2
print(f"a1 = {a1:.4f} kg·m²")
print(f"a2 = {a2:.4f} kg·m²")
print(f"a3 = {a3:.4f} kg·m²")

# Reference posture
q_ref = np.array([np.radians(55), np.radians(75)])
print(f"\nReference posture: θ₁ = 55°, θ₂ = 75°")
print("Setup complete.")

### Paste Your Classes from Labs 01 and 02
Paste your `Arm` class (Lab 01) and your `Muscle` class with supporting functions (Lab 02) below. Your `force_length` function should include the neural drive gain G = 20 mm from Lab 02, which scales dimensionless activations to physiologically realistic forces.

In [ ]:
# PASTE your Arm class and Muscle class here



---
## Part 1: The Mass-Inertia Matrix M(q) (10 pts)

The mass-inertia matrix M(q) encodes how inertia is distributed across the two joints. From the lecture notes (Eq. 3.6):

M₁₁ = a₁ + 2a₃ cosθ₂,  M₁₂ = M₂₁ = a₂ + a₃ cosθ₂,  M₂₂ = a₂

### Task 1.1 — Implement `mass_matrix(q)` (5 pts)

COMPLETE the function below. It should return a 2×2 numpy array.

In [ ]:
def mass_matrix(q):
    """Compute the 2x2 mass-inertia matrix M(q).
    
    Parameters:
        q: array [theta1, theta2] in radians
    Returns:
        M: 2x2 numpy array
    """
    theta2 = q[1]
    
    # TODO: COMPLETE — compute each element
    M11 = ...  # a1 + 2*a3*cos(theta2)
    M12 = ...  # a2 + a3*cos(theta2)
    M22 = ...  # a2
    
    return np.array([[M11, M12], [M12, M22]])

### Task 1.2 — Verify at θ₂ = 75° (5 pts)

COMPLETE the cell below to evaluate M at the reference posture and verify against the lecture notes (Section E of the supplementary):
- M₁₁ ≈ 0.342
- M₁₂ ≈ 0.087
- M₂₂ ≈ 0.064

In [ ]:
# TODO: COMPLETE — evaluate and print
M = mass_matrix(q_ref)
print("M(q) at θ₁=55°, θ₂=75°:")
print(f"  M11 = {M[0,0]:.4f}  (expected ≈ 0.342)")
print(f"  M12 = {M[0,1]:.4f}  (expected ≈ 0.087)")
print(f"  M22 = {M[1,1]:.4f}  (expected ≈ 0.064)")
print(f"  det(M) = {np.linalg.det(M):.5f}")
print(f"  Symmetric: {np.allclose(M, M.T)}")

---
## Part 2: Coriolis and Centrifugal Terms (10 pts)

All velocity-dependent forces come from a single function h = −a₃ sinθ₂. From the lecture notes (Eq. 4.1):

- Shoulder: h(dθ₂/dt)² + 2h(dθ₁/dt)(dθ₂/dt)
- Elbow: −h(dθ₁/dt)²

### Task 2.1 — Implement `coriolis(q, qdot)` (5 pts)

COMPLETE the function below. It should return a 2-element array (the C(q,dq/dt)·dq/dt vector).

In [ ]:
def coriolis(q, qdot):
    """Compute the Coriolis/centrifugal force vector C(q,qdot)*qdot.
    
    Parameters:
        q: array [theta1, theta2] in radians
        qdot: array [dtheta1/dt, dtheta2/dt] in rad/s
    Returns:
        c: 2-element array [c1, c2]
    """
    theta2 = q[1]
    dq1, dq2 = qdot
    
    # TODO: COMPLETE — compute h and the two terms
    h = ...  # -a3 * sin(theta2)
    
    c1 = ...  # h*dq2**2 + 2*h*dq1*dq2
    c2 = ...  # -h*dq1**2
    
    return np.array([c1, c2])

### Task 2.2 — Verify (5 pts)

COMPLETE the cell below to check that coriolis returns zero when velocities are zero, and nonzero otherwise.

In [ ]:
# Zero velocity → zero Coriolis
c_zero = coriolis(q_ref, np.array([0.0, 0.0]))
print(f"C at zero velocity: {c_zero}  (should be [0, 0])")

# Nonzero velocity
qdot_test = np.array([1.0, 2.0])
c_test = coriolis(q_ref, qdot_test)
h = -a3 * np.sin(q_ref[1])
print(f"\nh = {h:.4f}")
print(f"C at dq/dt=[1,2]: [{c_test[0]:.4f}, {c_test[1]:.4f}]")
print(f"  Expected c1 = h*4 + 2*h*2 = {h*4 + 2*h*2:.4f}")
print(f"  Expected c2 = -h*1 = {-h:.4f}")

---
## Part 3: Solve for Joint Accelerations (15 pts)

Given M and C, the joint accelerations are:

d²q/dt² = M(q)⁻¹ [τ − C(q,dq/dt)·dq/dt]

### Task 3.1 — Implement `joint_accelerations(q, qdot, tau)` (5 pts)

COMPLETE the function below.

In [ ]:
def joint_accelerations(q, qdot, tau):
    """Compute joint accelerations from the equations of motion.
    
    Parameters:
        q: array [theta1, theta2] in radians
        qdot: array [dtheta1/dt, dtheta2/dt] in rad/s
        tau: array [tau1, tau2] joint torques in N·m
    Returns:
        qddot: array [d2theta1/dt2, d2theta2/dt2] in rad/s²
    """
    M = mass_matrix(q)
    c = coriolis(q, qdot)
    
    # TODO: COMPLETE — solve M * qddot = tau - c
    qddot = ...  # np.linalg.solve(M, tau - c)
    
    return qddot

### Task 3.2 — Reproduce Lecture Figure 4: Shoulder-Only Torque (5 pts)

At the reference posture with zero velocity, apply τ = [1, 0] (shoulder only). COMPLETE the cell to verify:
- d²θ₁/dt² ≈ 4.49 rad/s² (shoulder accelerates forward)
- d²θ₂/dt² ≈ −6.11 rad/s² (elbow accelerates BACKWARD)

In [ ]:
# Shoulder-only torque
qddot_shoulder = joint_accelerations(q_ref, np.array([0., 0.]), np.array([1., 0.]))
print("Shoulder-only τ = [1, 0]:")
print(f"  d²θ₁/dt² = {qddot_shoulder[0]:.3f} rad/s²  (expected ≈ 4.49)")
print(f"  d²θ₂/dt² = {qddot_shoulder[1]:.3f} rad/s²  (expected ≈ -6.11)")
print(f"  → Elbow accelerates BACKWARD even though τ₂ = 0!")

# TODO: COMPLETE — also test elbow-only torque τ = [0, 1]
qddot_elbow = joint_accelerations(q_ref, np.array([0., 0.]), np.array([0., 1.]))
print(f"\nElbow-only τ = [0, 1]:")
print(f"  d²θ₁/dt² = {qddot_elbow[0]:.3f} rad/s²  (expected ≈ -6.11)")
print(f"  d²θ₂/dt² = {qddot_elbow[1]:.3f} rad/s²  (expected ≈ 23.89)")
print(f"  → Shoulder accelerates BACKWARD even though τ₁ = 0!")

### Question 3.1 (5 pts)

Why is |d²θ₂/dt²| larger than |d²θ₁/dt²| in the shoulder-only case? And why is the elbow acceleration so much larger (23.9 vs 4.5 rad/s²) in the elbow-only case? (Hint: compare M₁₁ to M₂₂.)

*Your answer here:*


---
## Part 4: Runge–Kutta 4th-Order Integrator (10 pts)

### Task 4.1 — Implement Generic RK4 Step (5 pts)

COMPLETE the generic RK4 integrator below. It takes a state vector, a derivative function, and a timestep.

In [ ]:
def rk4_step(state, deriv_fn, dt):
    """One step of 4th-order Runge-Kutta integration.
    
    Parameters:
        state: current state vector (numpy array)
        deriv_fn: function that takes state and returns dstate/dt
        dt: timestep
    Returns:
        new_state: updated state vector
    """
    # TODO: COMPLETE — implement RK4
    k1 = dt * deriv_fn(state)
    k2 = dt * deriv_fn(state + k1/2)
    k3 = dt * deriv_fn(state + k2/2)
    k4 = dt * deriv_fn(state + k3)
    
    return state + (k1 + 2*k2 + 2*k3 + k4) / 6

### Task 4.2 — Test on a Spring-Mass System (included in above)

Before using RK4 on the arm, test it on a simple system: a spring-mass oscillator with known solution.

x'' = -ω²x, with ω = 2π (period = 1 second). State = [x, dx/dt].

COMPLETE the cell below and verify the period is approximately 1 second.

In [ ]:
omega = 2 * np.pi

def spring_deriv(state):
    x, v = state
    return np.array([v, -omega**2 * x])

# Simulate 2 seconds
dt = 0.001; T = 2.0
t = np.arange(0, T, dt)
state = np.array([1.0, 0.0])  # x=1, v=0

x_traj = np.zeros(len(t))
for i in range(len(t)):
    x_traj[i] = state[0]
    # TODO: COMPLETE — advance one RK4 step
    state = rk4_step(state, spring_deriv, dt)

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, x_traj, '#2E86AB', lw=2, label='RK4')
ax.plot(t, np.cos(omega * t), 'k--', lw=1, alpha=0.5, label='Exact: cos(ωt)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('x')
ax.set_title('RK4 Test: Spring-Mass Oscillator')
ax.legend(); plt.show()

# Check error
error = np.max(np.abs(x_traj - np.cos(omega * t)))
print(f"Max error over 2 seconds: {error:.2e}  (should be < 1e-8)")

### Task 4.3 — Test RK4 on the Arm (5 pts)

Now test the arm with a constant torque τ = [0.5, 0] for 100 ms. State = [θ₁, θ₂, dθ₁/dt, dθ₂/dt]. COMPLETE the derivative function and simulate.

In [ ]:
def arm_deriv_simple(state, tau):
    """Derivative function for the arm with constant torque (no muscles yet)."""
    q = state[:2]
    qdot = state[2:]
    
    # TODO: COMPLETE — compute accelerations
    qddot = joint_accelerations(q, qdot, tau)
    
    return np.array([qdot[0], qdot[1], qddot[0], qddot[1]])

# Simulate
dt = 0.0001; T = 0.1
t = np.arange(0, T, dt)
state = np.array([np.radians(55), np.radians(75), 0.0, 0.0])
tau_const = np.array([0.5, 0.0])

q1_traj = np.zeros(len(t)); q2_traj = np.zeros(len(t))
for i in range(len(t)):
    q1_traj[i] = np.degrees(state[0])
    q2_traj[i] = np.degrees(state[1])
    state = rk4_step(state, lambda s: arm_deriv_simple(s, tau_const), dt)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t*1000, q1_traj, '#E74C3C', lw=2.5, label='θ₁ (shoulder)')
ax.plot(t*1000, q2_traj, '#3498DB', lw=2.5, label='θ₂ (elbow)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Angle (°)')
ax.set_title('Arm with Constant Shoulder Torque (no muscles)')
ax.legend(); plt.show()

print(f"After 100 ms: θ₁ = {q1_traj[-1]:.1f}° (started at 55°)")
print(f"              θ₂ = {q2_traj[-1]:.1f}° (started at 75°, should decrease)")

---
## Part 5: Connect the 6 Muscles (20 pts)

### Task 5.1 — Muscle Length and Velocity from Joint State (5 pts)

Each muscle's length depends on the joint angles via the moment arms (from Lab 02):

l_i ≈ l_i,rest − r_i,shoulder · (θ₁ − θ₁,ref) − r_i,elbow · (θ₂ − θ₂,ref)

dl_i/dt = −r_i,shoulder · dθ₁/dt − r_i,elbow · dθ₂/dt

COMPLETE the functions below.

In [ ]:
def muscle_length(muscle, q):
    """Compute muscle length from joint angles."""
    # TODO: COMPLETE
    dq1 = q[0] - q_ref[0]
    dq2 = q[1] - q_ref[1]
    return muscle.rest_length - muscle.moment_arm_shoulder * dq1 - muscle.moment_arm_elbow * dq2

def muscle_velocity(muscle, qdot):
    """Compute muscle velocity from joint velocities."""
    # TODO: COMPLETE
    return -muscle.moment_arm_shoulder * qdot[0] - muscle.moment_arm_elbow * qdot[1]

### Task 5.2 — Instantiate All 6 Muscles (5 pts)

COMPLETE the cell below to create the 6 muscles with the correct parameters from Lab 02. Use the same sign convention: positive moment arms for flexors, negative for extensors.

In [ ]:
# Muscle parameters from Lab 02 (Gribble et al. 1998)
# Format: Muscle(name, rho, k, moment_arm_shoulder, moment_arm_elbow, rest_length)

# TODO: COMPLETE — create all 6 muscles
# Flexors (positive moment arms)
pec = Muscle('pectoralis',      14.9, 258.5,  0.04,  0.0,  0.26)
bic_long = Muscle('biceps_long',11.0, 150.0,  0.0,   0.03, 0.26)
bic_short = Muscle('biceps_short',2.1, 100.0, 0.025, 0.03, 0.29)

# Extensors (negative moment arms)
deltoid = Muscle('deltoid',     14.9, 258.5, -0.04,  0.0,  0.26)
tri_lat = Muscle('triceps_lat', 12.1, 200.0,  0.0,  -0.02, 0.26)
tri_long = Muscle('triceps_long',6.7, 100.0, -0.04, -0.02, 0.32)

muscles = [pec, bic_long, bic_short, deltoid, tri_lat, tri_long]
print(f"Created {len(muscles)} muscles:")
for m in muscles:
    print(f"  {m.name}: r_sh={m.moment_arm_shoulder:+.3f}, r_el={m.moment_arm_elbow:+.3f}")

### Task 5.3 — Compute Joint Torques from Activations (10 pts)

COMPLETE the function below. For each muscle: compute length and velocity from q and qdot, call compute_force, then sum force × moment arm for each joint.

In [ ]:
def compute_torques(muscles, q, qdot, activations, dt):
    """Compute net joint torques from all 6 muscles.
    
    Parameters:
        muscles: list of 6 Muscle objects
        q: joint angles [theta1, theta2]
        qdot: joint velocities [dtheta1/dt, dtheta2/dt]
        activations: array of 6 activation levels
        dt: timestep for muscle integration
    Returns:
        tau: array [tau1, tau2] joint torques
    """
    tau = np.array([0.0, 0.0])
    
    for i, muscle in enumerate(muscles):
        # TODO: COMPLETE — compute length, velocity, force, torque contribution
        length = muscle_length(muscle, q)
        velocity = muscle_velocity(muscle, qdot)
        force = muscle.compute_force(activations[i], length, velocity, dt)
        
        tau[0] += muscle.moment_arm_shoulder * force
        tau[1] += muscle.moment_arm_elbow * force
    
    return tau

### Task 5.4 — Verify: Net Torque at Rest (included in points above)

At the reference posture with zero activation and zero velocity, the net torque should be approximately zero (only passive spring forces, which cancel at rest).

In [ ]:
# Reset all muscles
for m in muscles:
    m.reset()

# Compute torque at rest with zero activation
a_zero = np.zeros(6)
qdot_zero = np.zeros(2)
tau_rest = compute_torques(muscles, q_ref, qdot_zero, a_zero, 0.0001)
print(f"Net torque at rest (zero activation): τ = [{tau_rest[0]:.6f}, {tau_rest[1]:.6f}] N·m")
print(f"Should be ≈ [0, 0]")

---
## Part 6: Assemble the DynamicsEngine (10 pts)

Now wrap everything into a single class.

**Important design decision:** The muscles and the kinematics use different integration strategies:
- **Muscles** update once per timestep via Euler (inside `compute_force`). This is adequate because dt = 0.1 ms is much smaller than the calcium time constant τ = 15 ms.
- **Kinematics** are integrated with RK4 for accuracy. But RK4 evaluates the derivative four times per step. If we called the muscles inside the RK4 derivative function, the calcium state would advance four times instead of once — corrupting the muscle dynamics.

**Solution:** Compute muscle torques **once** at the start of each timestep, then pass the fixed torque vector to RK4. The `_kinematic_derivatives` function only computes M⁻¹[τ − C] — it never touches the muscles.

### Task 6.1 — COMPLETE the DynamicsEngine Class (10 pts)

In [ ]:
class DynamicsEngine:
    """Forward dynamics engine for the 2-link arm with 6 muscles."""
    
    def __init__(self, arm, muscles):
        self.arm = arm
        self.muscles = muscles
        self.n_muscles = len(muscles)
        
        # State: [theta1, theta2, dtheta1/dt, dtheta2/dt]
        self.state = np.array([np.radians(55), np.radians(75), 0.0, 0.0])
    
    def reset(self, q=None, qdot=None):
        """Reset to a given posture (or default reference)."""
        if q is None:
            q = np.array([np.radians(55), np.radians(75)])
        if qdot is None:
            qdot = np.zeros(2)
        self.state = np.concatenate([q, qdot])
        for m in self.muscles:
            m.reset()
    
    def _kinematic_derivatives(self, state, tau):
        """Compute kinematic state derivatives for a FIXED torque vector.
        
        This is the function RK4 integrates. It does NOT call the muscles —
        torques are computed once per timestep, then held constant during
        the four RK4 sub-evaluations.
        """
        q = state[:2]
        qdot = state[2:]
        
        # TODO: COMPLETE — compute accelerations from the pre-computed tau
        qddot = joint_accelerations(q, qdot, tau)
        
        return np.array([qdot[0], qdot[1], qddot[0], qddot[1]])
    
    def step(self, activations, dt):
        """Advance one timestep.
        
        Integration strategy:
        1. Muscles update ONCE (Euler, inside compute_force) to get tau
        2. RK4 integrates the kinematic state with tau held constant
        
        This avoids advancing the muscle calcium state four times per
        RK4 step. The muscles are stiff (tau_calcium = 15 ms) but slow
        relative to a single 0.1 ms timestep, so Euler is adequate for
        the muscle sub-step.
        """
        q = self.state[:2]
        qdot = self.state[2:]
        
        # Step 1: Update muscles ONCE, get joint torques
        # TODO: COMPLETE — compute torques from current state
        tau = compute_torques(self.muscles, q, qdot, activations, dt)
        
        # Step 2: RK4 on kinematics only, with tau held constant
        # TODO: COMPLETE — use rk4_step with _kinematic_derivatives
        deriv_fn = lambda s: self._kinematic_derivatives(s, tau)
        self.state = rk4_step(self.state, deriv_fn, dt)
        
        return self.state.copy()
    
    def simulate(self, activation_fn, T, dt=0.0001):
        """Run a full simulation.
        
        Parameters:
            activation_fn: function(t) -> array of 6 activations
            T: total time in seconds
            dt: timestep
        Returns:
            t: time array
            states: array of shape (N, 4) — [theta1, theta2, dtheta1, dtheta2]
        """
        t = np.arange(0, T, dt)
        states = np.zeros((len(t), 4))
        
        for i in range(len(t)):
            states[i] = self.state
            activations = activation_fn(t[i])
            self.step(activations, dt)
        
        return t, states

# Test: create the engine
arm = Arm()
engine = DynamicsEngine(arm, muscles)
engine.reset()
print(f"DynamicsEngine created. Initial state: θ₁={np.degrees(engine.state[0]):.1f}°, θ₂={np.degrees(engine.state[1]):.1f}°")

---
## Part 7: "Flick" Dynamics — Interaction Torques in Action (10 pts)

In the horizontal plane with no gravity and no activation, an arm at rest stays at rest. But if we give one joint an initial velocity — like someone flicking your upper arm — the interaction torques cause the *other* joint to move.

### Task 7.1 — Shoulder Flick (5 pts)

COMPLETE the cell below to simulate the arm starting from rest but with an initial shoulder velocity of dθ₁/dt = 2 rad/s (elbow initially stationary). Use zero activation throughout. Plot both joint angles over 500 ms.

In [ ]:
engine.reset(q=q_ref, qdot=np.array([2.0, 0.0]))  # shoulder flick

# Zero activation for the whole simulation
def zero_activation(t):
    return np.zeros(6)

t, states = engine.simulate(zero_activation, T=0.5, dt=0.0001)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.plot(t*1000, np.degrees(states[:,0]), '#E74C3C', lw=2.5, label='θ₁ (shoulder)')
ax1.plot(t*1000, np.degrees(states[:,1]), '#3498DB', lw=2.5, label='θ₂ (elbow)')
ax1.axhline(55, color='#E74C3C', ls=':', alpha=0.3); ax1.axhline(75, color='#3498DB', ls=':', alpha=0.3)
ax1.set_ylabel('Angle (°)'); ax1.legend(); ax1.set_title('Shoulder Flick: dθ₁/dt = 2 rad/s, dθ₂/dt = 0')

ax2.plot(t*1000, states[:,2], '#E74C3C', lw=2, label='dθ₁/dt')
ax2.plot(t*1000, states[:,3], '#3498DB', lw=2, label='dθ₂/dt')
ax2.axhline(0, color='gray', lw=0.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Velocity (rad/s)'); ax2.legend()
plt.tight_layout(); plt.show()

print(f"Shoulder: {np.degrees(states[0,0]):.1f}° → peak {np.degrees(np.max(states[:,0])):.1f}°")
print(f"Elbow: {np.degrees(states[0,1]):.1f}° → min {np.degrees(np.min(states[:,1])):.1f}°, max {np.degrees(np.max(states[:,1])):.1f}°")

### Task 7.2 — Elbow Flick (5 pts)

Now do the reverse: initial elbow velocity of dθ₂/dt = 3 rad/s, shoulder stationary. COMPLETE and plot.

In [ ]:
# TODO: COMPLETE — simulate elbow flick
engine.reset(q=q_ref, qdot=np.array([0.0, 3.0]))  # elbow flick

t, states = engine.simulate(zero_activation, T=0.5, dt=0.0001)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.plot(t*1000, np.degrees(states[:,0]), '#E74C3C', lw=2.5, label='θ₁ (shoulder)')
ax1.plot(t*1000, np.degrees(states[:,1]), '#3498DB', lw=2.5, label='θ₂ (elbow)')
ax1.axhline(55, color='#E74C3C', ls=':', alpha=0.3); ax1.axhline(75, color='#3498DB', ls=':', alpha=0.3)
ax1.set_ylabel('Angle (°)'); ax1.legend(); ax1.set_title('Elbow Flick: dθ₁/dt = 0, dθ₂/dt = 3 rad/s')

ax2.plot(t*1000, states[:,2], '#E74C3C', lw=2, label='dθ₁/dt')
ax2.plot(t*1000, states[:,3], '#3498DB', lw=2, label='dθ₂/dt')
ax2.axhline(0, color='gray', lw=0.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Velocity (rad/s)'); ax2.legend()
plt.tight_layout(); plt.show()

print(f"Shoulder: {np.degrees(states[0,0]):.1f}° → was it dragged?")
print(f"Elbow: {np.degrees(states[0,1]):.1f}° → peak {np.degrees(np.max(states[:,1])):.1f}°")

---
## Part 8: Muscle-Driven Movement (15 pts)

### Task 8.1 — Activate Shoulder Flexors Only (5 pts)

COMPLETE the cell below to simulate 300 ms with only the pectoralis (muscle 0) activated at a = 0.5. All other muscles at zero. This is the muscle-driven version of the lecture's interaction torque demonstration.

In [ ]:
engine.reset()

def shoulder_flexor_only(t):
    a = np.zeros(6)
    if t >= 0.02:  # command at 20 ms
        a[0] = 0.5  # pectoralis
    return a

t, states = engine.simulate(shoulder_flexor_only, T=0.3, dt=0.0001)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t*1000, np.degrees(states[:,0]), '#E74C3C', lw=2.5, label='θ₁ (shoulder)')
ax.plot(t*1000, np.degrees(states[:,1]), '#3498DB', lw=2.5, label='θ₂ (elbow)')
ax.axhline(55, color='#E74C3C', ls=':', alpha=0.3); ax.axhline(75, color='#3498DB', ls=':', alpha=0.3)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Angle (°)')
ax.set_title('Shoulder Flexor Only (pectoralis a=0.5)')
ax.legend(); plt.show()

print(f"Shoulder: 55° → {np.degrees(states[-1,0]):.1f}° (flexed)")
print(f"Elbow: 75° → {np.degrees(states[-1,1]):.1f}° (should have moved too!)")

### Task 8.2 — Shoulder Flexion + Elbow Co-Contraction (5 pts)

Now activate the pectoralis AND co-contract the elbow (biceps long + triceps lateral). Does the elbow resist the interaction torque?

COMPLETE the activation function.

In [ ]:
engine.reset()

def shoulder_with_elbow_cocontraction(t):
    a = np.zeros(6)
    if t >= 0.02:
        a[0] = 0.5  # pectoralis (shoulder flexor)
        a[1] = 0.3  # biceps long (elbow flexor)
        a[4] = 0.3  # triceps lateral (elbow extensor)
    return a

t2, states2 = engine.simulate(shoulder_with_elbow_cocontraction, T=0.3, dt=0.0001)

fig, ax = plt.subplots(figsize=(10, 5))
# Plot both conditions
ax.plot(t*1000, np.degrees(states[:,1]), '#3498DB', lw=1.5, ls='--', alpha=0.6, label='θ₂ without co-contraction')
ax.plot(t2*1000, np.degrees(states2[:,1]), '#3498DB', lw=2.5, label='θ₂ with elbow co-contraction')
ax.axhline(75, color='gray', ls=':', alpha=0.3)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Elbow angle (°)')
ax.set_title('Effect of Elbow Co-Contraction on Interaction Torque')
ax.legend(); plt.show()

delta_no_cocon = np.degrees(states[-1,1]) - 75
delta_cocon = np.degrees(states2[-1,1]) - 75
print(f"Elbow displacement without co-contraction: {delta_no_cocon:.1f}°")
print(f"Elbow displacement with co-contraction: {delta_cocon:.1f}°")
print(f"Reduction: {(1 - abs(delta_cocon)/abs(delta_no_cocon))*100:.0f}%")

### Question 8.1 (5 pts)

**(a)** Does elbow co-contraction reduce the interaction torque effect? By approximately what percentage?

**(b)** What is the cost of this strategy? (Hint: both elbow muscles are active but producing opposing torques.)

**(c)** The lecture notes describe co-contraction as a "brute-force" solution. In Week 4, the λ model will provide a more elegant alternative. Based on what you know so far, what would an "elegant" solution look like?

*Your answer here:*


---
## Summary

You built the DynamicsEngine — the bridge between muscles and movement:

- **M(q):** The mass-inertia matrix. Configuration-dependent (cosθ₂ coupling). Three constants a₁, a₂, a₃ define everything.
- **C(q,dq/dt):** Coriolis and centrifugal terms. All from a single function h = −a₃ sinθ₂.
- **RK4 integration:** 4th-order accurate, stable for coupled muscle + limb dynamics.
- **Interaction torques:** Torque at one joint produces acceleration at both. Demonstrated with flick dynamics and muscle-driven movement.
- **Co-contraction:** Resists interaction torques but at metabolic cost.

**Next week:** The λ threshold law replaces direct activation a(t) with a centrally specified threshold λ(t). Activation becomes A = [l − λ]⁺ — dependent on muscle length. This creates automatic length-dependent stiffness that resists perturbations without the cost of co-contraction. The DynamicsEngine you built this week is the plant that all future controllers will drive.